# 量子控制实验主程序

使用QDriver三层架构进行量子测量

- Layer 1: 设备管理层 (driver) - 硬件初始化和控制
- Layer 2: 时序层 (experiment) - 测量时序函数
- Layer 3: 数据管理层 (datacollector) - 结果处理和存储

## Part 1: 初始化硬件系统

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from tqdm.notebook import tqdm
from IPython.display import clear_output

# 导入QDriver框架
from qdriver.driver import initialize_system, reset_device, XYDrive, Readout
from qdriver.experiment import measure_cavity_frequency
from qdriver.datacollector import (
    ExperimentContext,
    setup_qcodes_experiment,
    plot_cavity_resonance
)

In [ ]:
# 初始化硬件系统
device, qsync, darate, adrate = initialize_system(
    device_ip="192.168.1.217",
    qsync_ip="192.168.1.217",
    debug=True
)

In [ ]:
# 创建XY驱动器（用于S5-O1通道）
xy_drive = XYDrive(
    device=device,
    qsync=qsync,
    channel="S5-O1",
    darate=darate
)

# 创建读出器（用于S9-I1输入，S9-O1输出）
readout = Readout(
    device=device,
    qsync=qsync,
    input_channel="S9-I1",
    output_channel="S9-O1",
    adrate=adrate
)

# 创建实验上下文
exp_ctx = ExperimentContext()

print("设备创建完成")

## Part 2: 配置QCoDeS实验

In [ ]:
import os
from qcodes.instrument_drivers.SIMIT.DataLogger1 import DataLogger
import qcodes as qc

# 配置QCoDeS数据库
db_filename = 'experiment_20260330.db'
db_file_path = os.path.join('D:\\Test_data\\naishu', db_filename)

exp, meas, station = setup_qcodes_experiment(
    exp_name="Cavity_Resonance",
    sample_name="Sample_001",
    db_path=db_file_path,
    station=None
)

# 添加数据记录器
DL1 = DataLogger("DL1")
station.add_component(DL1)
DL2 = DataLogger("DL2")
station.add_component(DL2)
DL3 = DataLogger("DL3")
station.add_component(DL3)
DL4 = DataLogger("DL4")
station.add_component(DL4)

# 注册测量参数
meas.register_parameter(DL1.frequency)
meas.register_parameter(DL1.magnitude, setpoints=(DL1.frequency,))
meas.register_parameter(DL1.phase, setpoints=(DL1.frequency,))

print("QCoDeS配置完成")

## Part 3: 配置XY驱动器和读出器

In [ ]:
# 配置XY驱动器
# 单个矩形脉冲，10ns宽度，幅度0.5，频率5GHz
xy_drive.configure(
    shape_list=['Square'],
    pw_list=[10e-9],                 # 脉冲宽度
    ew_list=[1e-9],                  # 包络宽度
    amp_list=[0.5],                  # 幅度 (0-1)
    center_list=[0],                 # 中心位置
    phase_list=[0],                  # 初始相位 (度)
    freq_list=[5e9],                 # 初始频率 (Hz)
    delay=100e-6,                    # 脉冲前置延迟
    spacing_time=24e-9               # 多脉冲间隔
)

# 配置读出器
readout.configure(
    delay=100e-6,                    # 主延迟
    in_delay=200e-9,                 # 输入延迟
    sample_length=4096e-9,           # 采集长度
    shots=1024*10,                   # 重复次数
    period=200e-6                    # Trigger周期
)

# 设置读出频率和幅度
readout.set_frequency(5e9)           # 读出频率
readout.set_amplitude(0.2)           # 读出幅度

print("设备配置完成")

## Part 4: 腔频测量与数据存储

In [ ]:
# 定义扫频参数
center_freq = 6.26e9  # 扫频中心
span = 4e6            # 扫频跨度 (Hz)
num_points = 41       # 扫频点数

freq_list = np.linspace(center_freq - span/2, center_freq + span/2, num_points)

print(f"扫频范围: {freq_list[0]/1e9:.3f} GHz 到 {freq_list[-1]/1e9:.3f} GHz")
print(f"扫频点数: {num_points}")

# 执行腔频测量
cavity_result = measure_cavity_frequency(
    xy_drive=xy_drive,
    readout=readout,
    freq_list=freq_list
)

print(f"\n测量完成，共 {len(cavity_result['mag'])} 个数据点")

# 自动存储到QCoDeS数据库
with meas.run() as datasaver:
    for freq, mag, phase in zip(cavity_result['freq'], cavity_result['mag'], cavity_result['phase']):
        datasaver.add_result(
            (DL1.frequency, freq),
            (DL1.magnitude, mag),
            (DL1.phase, phase)
        )

dataset = datasaver.dataset
print(f"数据已保存到QCoDeS，Run ID: {dataset.run_id}")

## Part 5: 数据分析与绘图

In [ ]:
# 从测量数据中提取腔参数
exp_ctx.update_from_cavity_scan(
    freq_array=cavity_result['freq'],
    mag_array=cavity_result['mag'],
    phase_array=cavity_result['phase']
)

# 获取腔参数
cavity_params = exp_ctx.get_cavity_params()

print("\n腔参数提取结果：")
print(f"  腔频: {cavity_params['freq']/1e9:.6f} GHz")
print(f"  品质因数 Q: {cavity_params['q']:.0f}")
print(f"  线宽: {cavity_params['linewidth']/1e6:.3f} MHz")

# 绘制腔频共振曲线
fig = plot_cavity_resonance(cavity_result, cavity_params)
plt.show()

## Part 6: 清理和关闭

In [ ]:
# 设置设备到安全状态
xy_drive.zero()          # XY驱动器输出置零
readout.set_shots(0)     # 停止读出

# 重置硬件系统
reset_device(device, qsync)

print("设备已置零并重置")